# Step 1 - Initialization
1. Initial setup
2. Initialized model architecture
3. initialize training dataset


In [ ]:
# step 1.1
import torch

# Set device (GPU if available)
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [ ]:
# step 1.2
from model import SimpleCNN

model = SimpleCNN().to(device)

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load datasets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Step 2 - Training

In [ ]:
# setup monitoring
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir="./runs/mnist_cnn")


In [ ]:
# define training function 
import torch.nn.functional as F

def train(model, device, train_loader, optimizer, epoch, writer):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()

        global_step = epoch * len(train_loader) + batch_idx
        if batch_idx % 100 == 0:
            writer.add_scalar("Loss/train", loss.item(), global_step)
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}')


3. Epoch Loop

In [ ]:
%%time 
import torch.optim as optim

# from utils.validate import validate

optimizer = optim.Adadelta(model.parameters(), lr=1.0)

# Train 5 epoch
epochs = 10
for epoch in range(0, epochs):
    train(model, device, train_loader, optimizer, epoch, writer)
    # validate(model, device, test_loader)  # 可选：每个 epoch 后验证